# 05 – SVM tuyến tính: ablation đặc trưng, tinh chỉnh C, kiểm chứng Group split

**Người thực hiện:** danh (Huỳnh Quốc Danh)  
**Mục tiêu:** Trả lời RQ1 và chọn mô hình cuối cùng cho hệ thống.

**Quy tắc:** chạy TỪNG ô một, đổi tham số rồi chạy lại; mỗi ngày làm xong thì `git add` + `git commit -m "<mô tả đúng việc đã làm>"` + `git push`. Cuối buổi xuất HTML: `python ../scripts/export_notebook_html.py <notebook> danh`.

**Các bước:**
1. Ablation: word → char → hybrid → full (cùng C).
2. Thử char_ngram (2,4), (3,5), (3,6).
3. Quét C ∈ {0.03, 0.1, 0.3, 1, 3} có CV.
4. class_weight='balanced' có giúp 3 lớp tủ lạnh/tủ đông?
5. Chạy lại cấu hình tốt nhất với split='group'.
6. GridSearchCV nhỏ (tùy chọn) và kết luận mô hình chọn.

In [9]:
import sys, json, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.append('../src')
from run_experiment import run
from figlog import save_fig
from exp_logger import log_generic
from common import load_data, clean_title


## 1. Ablation đặc trưng

In [10]:
res, _ = run(member='danh', model='svc', features='full', params={'C': 1.0}, exp_name='abl_full', show_report=False)

[danh_20261112_133323] svc/full {'C': 1.0} split=strat
   train_time_s                 29.7630
   test_accuracy                0.9925
   test_precision_macro         0.9923
   test_recall_macro            0.9916
   test_f1_macro                0.9920
   test_f1_weighted             0.9925
   f1_Fridges                   0.9812
   f1_Freezers                  0.9749
   f1_Fridge_Freezers           0.9914
→ Đã ghi vào experiments/danh/experiment_log.csv. Hãy mở file và điền cột nhan_xet.


> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/danh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính:full, C1 → Acc 0.9925, Macro 0.9920; F1 Fridges 0.9812, Freezers 0.9749, Fridge Freezers 0.9914; train 29.8s
> - Nhận xét: 
1. So với word(0.9679) full tăng 0,0241 tăng khoảng 2.4%. Nghe thì nhỏ, nhưng tính theo lỗi: tập Test có 7.063 mẫu, F1 từ ~96,8% lên ~99,2% nghĩa là số mẫu đoán sai giảm khoảng 4 lần. Giữa kỳ đếm được: word sai 234 mẫu, hybrid chỉ còn 55 mẫu.
2. So với hybird, full tăng 0.003 tức 0.03%. Mức này nhỏ hơn dao động giữa các fold khi chạy CV. Nói cách khác, chênh lệch này có thể chỉ do may rủi khi chia dữ liệu. Vậy gần như toàn bộ cải thiện đến từ việc thêm n-gram ký tự. Phần hình thái + merchant chưa chứng minh được là có ích.
Lý do n-gram ký tự giúp nhiều: khoảng 1/5 tiêu đề không có từ khóa ngành, chỉ có hãng + mã model như kif52af30, hoặc là chữ tiếng Đức bị mất dấu như k hlschrank. N-gram theo từ coi cả mã là một “từ lạ” nên không học được. N-gram ký tự tách thành các mảnh như kif, f52, hls…, nhờ đó học được rằng tiền tố kif của Bosch thường là tủ lạnh.
3. Freezers thấp nhất (0.9749) vì:
Ít mẫu nhất: chỉ 2.212 mẫu (6,26%), ít hơn 2,5 lần so với Fridge Freezers (5.501 mẫu). Mô hình có ít ví dụ để học lớp này hơn.
Từ vựng trùng nhiều: độ trùng từ vựng (Jaccard) giữa Freezers và Fridges là 0,27, cao nhất toàn bộ dữ liệu; với Fridge Freezers là 0,21. Các từ như “freestanding”, “white”, “integrated”, “a+” xuất hiện ở cả ba lớp. Thêm vào đó, cụm “fridge freezer” chứa cả hai chữ “fridge” lẫn “freezer”, nên mô hình dễ lẫn.

Lỗi thực tế ở bản giữa kỳ khớp với giải thích này: cặp nhầm nhiều nhất là Freezers → Fridges (8 mẫu), Fridges → Fridge Freezers (8 mẫu), Fridges → Freezers (5 mẫu).
4. Có mất nhiều thời gian quá không
Trên máy tôi, full mất 29,8 giây. Xem cột train_time_s của dòng tai_lap_E04 để so; trên máy mình hồi giữa kỳ, word chỉ mất khoảng 1 giây, full khoảng 10 giây. Vậy full chậm hơn nhiều lần mà chỉ hơn hybrid 0,03 điểm. hybrid là lựa chọn hợp lý hơn: nhanh hơn mà điểm gần như bằng. Còn 30 giây để huấn luyện vẫn chấp nhận được, vì chỉ huấn luyện một lần, còn lúc dự đoán thì rất nhanh.
> - Lần sau thử: 
Quét C (0.1, 0.3, 1, 3) cho hybrid, chạy kèm --cv để có độ lệch chuẩn, nhờ đó so sánh được một cách chắc chắn.
Thử class_weight='balanced' để xem Freezers có tăng không, vì đây là lớp ít mẫu.

Đổi `features` thành `'char'`, `'hybrid'`, `'full'` và chạy lại – mỗi lần ghi nhận xét.

## 2. Phạm vi n-gram ký tự

In [3]:
res, _ = run(member='danh', model='svc', features='hybrid', params={'C': 1.0}, char_ngram=(2, 4), exp_name='char_24', show_report=False)

[danh_20261112_124329] svc/hybrid {'C': 1.0} split=strat
   train_time_s                 14.2319
   test_accuracy                0.9929
   test_precision_macro         0.9927
   test_recall_macro            0.9921
   test_f1_macro                0.9924
   test_f1_weighted             0.9929
   f1_Fridges                   0.9818
   f1_Freezers                  0.9739
   f1_Fridge_Freezers           0.9918
→ Đã ghi vào experiments/danh/experiment_log.csv. Hãy mở file và điền cột nhan_xet.


> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/danh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 3. Quét C (có 5-fold CV)

In [4]:
res, _ = run(member='danh', model='svc', features='full', params={'C': 0.3}, cv=True, exp_name='C_0.3', show_report=False)

[danh_20261112_131842] svc/full {'C': 0.3} split=strat
   cv_f1_macro_mean             0.9866
   cv_f1_macro_std              0.0010
   cv_time_s                    2099.0941
   train_time_s                 11.5710
   test_accuracy                0.9904
   test_precision_macro         0.9905
   test_recall_macro            0.9895
   test_f1_macro                0.9900
   test_f1_weighted             0.9904
   f1_Fridges                   0.9709
   f1_Freezers                  0.9748
   f1_Fridge_Freezers           0.9891
→ Đã ghi vào experiments/danh/experiment_log.csv. Hãy mở file và điền cột nhan_xet.


> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/danh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 4. class_weight

In [5]:
res, _ = run(member='danh', model='svc', features='full', params={'C': 1.0, 'class_weight': 'balanced'}, exp_name='balanced')

[danh_20261112_131859] svc/full {'C': 1.0, 'class_weight': 'balanced'} split=strat
   train_time_s                 14.3775
   test_accuracy                0.9921
   test_precision_macro         0.9917
   test_recall_macro            0.9912
   test_f1_macro                0.9914
   test_f1_weighted             0.9921
   f1_Fridges                   0.9791
   f1_Freezers                  0.9727
   f1_Fridge_Freezers           0.9918
                  precision    recall  f1-score   support

            CPUs     0.9987    0.9987    0.9987       773
 Digital Cameras     0.9944    0.9944    0.9944       540
     Dishwashers     0.9913    0.9942    0.9927       685
        Freezers     0.9772    0.9683    0.9727       442
 Fridge Freezers     0.9918    0.9918    0.9918      1100
         Fridges     0.9804    0.9777    0.9791       717
      Microwaves     0.9979    0.9979    0.9979       468
   Mobile Phones     0.9951    1.0000    0.9976       816
             TVs     0.9972    0.9944    0

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/danh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 5. Group split theo Cluster ID
Câu hỏi phải trả lời được: *vì sao F1 giảm khi chia theo sản phẩm? giảm bao nhiêu là chấp nhận được?*

In [6]:
res, _ = run(member='danh', model='svc', features='full', params={'C': 1.0}, split='group', exp_name='group_best', show_report=False)

[danh_20261112_131917] svc/full {'C': 1.0} split=group
   train_time_s                 14.7156
   test_accuracy                0.9871
   test_precision_macro         0.9877
   test_recall_macro            0.9860
   test_f1_macro                0.9868
   test_f1_weighted             0.9871
   f1_Fridges                   0.9614
   f1_Freezers                  0.9761
   f1_Fridge_Freezers           0.9804
→ Đã ghi vào experiments/danh/experiment_log.csv. Hãy mở file và điền cột nhan_xet.


> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/danh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 6. Bảng so sánh & hình

In [7]:
log = pd.read_csv('../experiments/danh/experiment_log.csv')
log[['run_id','ten_thi_nghiem','features','params','split','test_f1_macro','f1_Fridges','f1_Freezers','nhan_xet']]

,run_id,ten_thi_nghiem,features,params,split,test_f1_macro,f1_Fridges,f1_Freezers,nhan_xet
0,danh_20261112_124307,abl_word,word,"{""C"": 1.0}",strat,0.967909,0.923398,0.953036,NaN
1,danh_20261112_124329,char_24,hybrid,"{""C"": 1.0}",strat,0.992425,0.981844,0.973893,NaN
2,danh_20261112_131842,C_0.3,full,"{""C"": 0.3}",strat,0.990015,0.970914,0.974828,NaN
3,danh_20261112_131859,balanced,full,"{""C"": 1.0, ""class_weight"": ""balanced""}",strat,0.991449,0.979050,0.972727,NaN
4,danh_20261112_131917,group_best,full,"{""C"": 1.0}",group,0.986821,0.961420,0.976067,NaN


## 7. Mô hình chọn và lý do (tự viết – đây là câu thầy sẽ hỏi)
- Đặc trưng: … vì …
- Mô hình: … vì …
- Tham số: … (dẫn run_id chứng minh)